In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_excel('/home/mahaputra971/pt_skp/staah/BookingsList_As_On_07-Oct-2026_1791368862.xlsx')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data columns (total 32 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Booking Date                   332 non-null    object 
 1   Modify Date                    332 non-null    object 
 2   Cancel Date                    88 non-null     object 
 3   Source                         332 non-null    object 
 4   Point Of Sale                  329 non-null    object 
 5   Status                         332 non-null    object 
 6   Room Type                      329 non-null    object 
 7   Package                        329 non-null    object 
 8   Arrival Date                   332 non-null    object 
 9   Departure Date                 332 non-null    object 
 10  Room(s)                        329 non-null    float64
 11  Total night(s)                 332 non-null    float64
 12  NoAdult                        332 non-null    flo

In [10]:
import pandas as pd
import numpy as np
import re
import openpyxl
from openpyxl.utils import get_column_letter

# 1. Bersihkan zona waktu pada Booking Date
booking_clean = df['Booking Date'].astype(str).str.replace(r'\(.*?\)', '', regex=True).str.strip()
booking_dt = pd.to_datetime(booking_clean, errors='coerce')

# 2. Bersihkan anomali pada Arrival Date (seperti -0001)
arrival_clean = df['Arrival Date'].astype(str).replace(r'.*-0001.*', np.nan, regex=True)
arrival_dt = pd.to_datetime(arrival_clean, errors='coerce')
arrival_dt = arrival_dt.mask(arrival_dt.dt.year < 2000)

# 3. Hitung Lead Time (dalam hari)
df['Lead Time (Days)'] = (arrival_dt.dt.normalize() - booking_dt.dt.normalize()).dt.days

# 4. Tentukan 6 kolom untuk sheet per-Source
selected_columns = [
    'Booking Date', 
    'Arrival Date', 
    'Status', 
    'Source', 
    'Lead Time (Days)', 
    'Total night(s)'
]

output_filename = "Data_Booking_Lead_Time_by_Source.xlsx"

# 5. Ekspor ke Excel
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    # Sheet 'All Data': Tetap menyimpan seluruh data asli lengkap
    df.to_excel(writer, sheet_name='All Data', index=False)
    
    # Sheet per masing-masing Source
    for source_name, group in df.groupby('Source', dropna=False):
        sheet_title = "Unknown" if pd.isna(source_name) else str(source_name).strip()
        clean_sheet_title = re.sub(r'[\:\\/\?\*\[\]]', '_', sheet_title)[:31]
        
        # Ambil hanya kolom terpilih
        group_filtered = group[selected_columns].copy()
        
        # --- HAPUS BARIS YANG LEAD TIME-NYA KOSONG / NaN ---
        group_filtered = group_filtered.dropna(subset=['Lead Time (Days)'])
        
        # Ubah format angka Lead Time menjadi integer (tanpa desimal .0)
        group_filtered['Lead Time (Days)'] = group_filtered['Lead Time (Days)'].astype(int)
        
        # Simpan ke sheet masing-masing
        group_filtered.to_excel(writer, sheet_name=clean_sheet_title, index=False)

# 6. (Opsional tapi Direkomendasikan) Rapikan Lebar Kolom agar Tidak Terpotong
wb = openpyxl.load_workbook(output_filename)
for sheet in wb.sheetnames:
    ws = wb[sheet]
    for col in ws.columns:
        max_length = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_length + 3, 12)
wb.save(output_filename)

print(f"Selesai! File berhasil diekspor tanpa baris kosong di: {output_filename}")

Selesai! File berhasil diekspor tanpa baris kosong di: Data_Booking_Lead_Time_by_Source.xlsx
